# Predicting Brain Pathology in Alzheimer's Disease using Elastic Net Regression

The notebook aims to check the data and prepare input for predictive models.

- Inputs:
    - Features (full_filled_year_dat_0303.txt): Clinical features include cognitive test scores, underlying health conditions, demographic.... 
    - Targets (pathology_dat_0303.txt): Pathology profiling includes amyloid, tangles, gpath, and NIA-Reagan scores.

- Outputs:
    - Prepare data from last visit for Elastic-Net
      - scaled_train_last.pkl, scaled_test_last.pkl
    - Prepare data from all visits for LSTM
      - scaled_train.pkl, scaled_test.pkl

# Analysis Steps
- Check Data
    - Correlation Analysis:
      - Identified 40 features with Pearson correlation coefficients > 0.4. Displayed results via a correlation heatmap.
- Data Preparation
    - Splitting data into 80% training and 20% testing sets.
    - Scaling features for model input.

# Set up

In [ ]:
import utils
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
import pandas as pd 
from sklearn.model_selection import train_test_split

In [ ]:
# Define paths
features_path = "Inputs/full_filled_year_dat_0303.txt"
targets_path = "Inputs/pathology_dat_with_modifiedNIA_050425.txt"
features_path_DL = "Inputs/full_filled_year_dat_0725.txt" 
targets_path_DL = "Inputs/pathology_dat_with_modifiedNIA_0725.txt"
data_DL = utils.load_and_merge_data(features_path_DL, targets_path_DL, join_columns=['projid', 'study'])
unique_ids_DL = data_DL['projid'].unique()


In [ ]:
data_DL.shape

In [ ]:
# Split subjects into training and testing data
train_ids_DL, test_ids_DL = train_test_split(unique_ids_DL, test_size=0.2, random_state=1217)

# Split the original dataframe into training and testing sets
train_data_DL = data_DL[data_DL['projid'].isin(train_ids_DL)]
test_data_DL = data_DL[data_DL['projid'].isin(test_ids_DL)]

In [ ]:
print('Training: ', len(train_ids_DL), 'Subject,', len(train_data_DL), 'Records') # Previous 1677 Subject, 16336 Records
print('Testing: ', len(test_ids_DL), 'Subject,', len(test_data_DL), 'Records') # Testing:  420 Subject, 4098 Records

In [ ]:
# Read the file，sep='\t' specifies tab-separated values
livings_msex_data = pd.read_csv(
    "Inputs/full_filled_year_dat_0303.txt",
    sep="\t",           # Tab-separated values
    index_col=0         # Use the first column as the index and exclude it from regular data columns
)

print(livings_msex_data.head())

# Python
livings_msex_data.to_csv("Outputs/livings_msex_data.csv", index=False, encoding="utf-8")

# 1. Check Data 

In [ ]:
# Load data
data = utils.load_and_merge_data(features_path, targets_path, join_columns=['projid', 'study'])

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
# number of subjects
unique_ids = data['projid'].unique()
len(unique_ids)

In [ ]:
print(data['niareagansc'].describe())

## 1.1 Check correlation between features

In [ ]:
exclude_columns = ['study', 'projid', 'fu_year', 'cogdx']
correlation_matrix = data.drop(exclude_columns, axis=1).corr()

# Create the initial mask for the upper triangle and values below 0.4
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool)) | (correlation_matrix.abs() < 0.4)

# Identify rows and columns that are fully masked
keep_rows = ~np.all(mask, axis=1)  # Rows to keep
keep_cols = ~np.all(mask, axis=0)  # Columns to keep

filtered_corr = correlation_matrix.loc[keep_rows, keep_cols]
filtered_mask = mask.loc[keep_rows, keep_cols]


- 40 features have correlation coefficent > 0.4 with at least one feature 

In [ ]:
combined_list = filtered_corr.index.to_list() + filtered_corr.columns.to_list()
len(set(combined_list))

- correlation heatmap

In [ ]:
# Filter the correlation matrix to include only rows and columns that are not fully masked
# Set up the figure size
plt.figure(figsize=(10, 8))

# Create the heatmap with the filtered correlation matrix and mask
sns.heatmap(
    filtered_corr,
    mask=filtered_mask,
    annot=filtered_corr[filtered_corr.abs() > 0.4],  # Annotate only values > 0.4
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    fmt='.2f',  # Format for annotations
    annot_kws={"size": 8},  # Annotation font size
    xticklabels=True,
    yticklabels=True,
)

# Rotate x-axis labels for better visibility
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

# Add a title to the heatmap
plt.title(r"Filtered Correlation Matrix Heatmap ($\rho$ > 0.4)", fontsize=16)

# Adjust layout to prevent cutoff
plt.tight_layout()

# Show the heatmap
plt.show()

## 1.2 Split data into training and testing

In [ ]:
# Split subjects into training and testing data
from sklearn.model_selection import train_test_split
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

# Split the original dataframe into training and testing sets
train_data = data[data['projid'].isin(train_ids)]
test_data = data[data['projid'].isin(test_ids)]

In [ ]:
print('Training: ', len(train_ids), 'Subject,', len(train_data), 'Records')
print('Testing: ', len(test_ids), 'Subject,', len(test_data), 'Records')

## 2. Prepare data from last visit for Elastic-Net

## 2.1 Get the last visit for training and testing datasets

In [ ]:
train_last = utils.get_last_visit(train_data, ['fu_year'], 'projid')
test_last = utils.get_last_visit(test_data, ['fu_year'], 'projid')

## 2.2 Scale data

In [ ]:
# Columns to scale
exclude_columns = ['study', 'projid', 'fu_year', 'cogdx' ]
columns_to_scale = [col for col in train_last.columns if col not in exclude_columns]

# Scale the last visit (training)
scaled_train_last, scalers = utils.transform_data(data=train_last, columns=columns_to_scale,
                                              transformer_type='StandardScaler', fit=True)
# Scale the last visit of testing using the same scalers
scaled_test_last = utils.transform_data(data=test_last, columns=columns_to_scale, 
                                    transformers=scalers, fit=False)

In [ ]:
scaled_train_last.to_pickle("Outputs/NIAvalueUpdatedVersion_scaled_train_last.pkl")
scaled_test_last.to_pickle("Outputs/NIAvalueUpdatedVersion_scaled_test_last.pkl")

## 3. Prepare input tensor data for LSTM

In [ ]:
# Define columns to scale (excluding 'study', 'projid', 'fu_year', 'cogdx')
columns_to_scale = train_data.drop(columns=['study', 'projid', 'fu_year', 'cogdx']).columns

# # Transform training data (fit and scale)
# scaled_train, scalers = utils.transform_data(
#     data=train_data,
#     columns=columns_to_scale,
#     transformer_type="MinMaxScaler",
#     fit=True,  # Fit new scalers
# )

# # Transform testing data (use pre-fitted scalers)
# scaled_test = utils.transform_data(
#     data=test_data,
#     columns=columns_to_scale,
#     transformer_type="MinMaxScaler",
#     transformers=scalers,  # Use pre-fitted scalers
#     fit=False,  # Do not fit new scalers
# )

# Transform 100% decedents data (fit and scale)
scaled_whole_decedents_dataset, scalers = utils.transform_data(
    data=data,
    columns=columns_to_scale,
    transformer_type="MinMaxScaler",
    fit=True,  # Fit new scalers
)

# Transform livings data (use pre-fitted scalers)
scaled_train_DL = utils.transform_data(
    data=train_data_DL,
    columns=columns_to_scale,
    transformer_type="MinMaxScaler",
    transformers=scalers,  # Use pre-fitted scalers
    fit=False,  # Do not fit new scalers
)

# Transform livings data (use pre-fitted scalers)
scaled_test_DL = utils.transform_data(
    data=test_data_DL,
    columns=columns_to_scale,
    transformer_type="MinMaxScaler",
    transformers=scalers,  # Use pre-fitted scalers
    fit=False,  # Do not fit new scalers
)

In [ ]:
print(f"Shape for the scale train_DL is {scaled_train_DL.shape}")
print(f"Shape for the scale test_DL is {scaled_test_DL.shape}")

In [ ]:
scaled_train_DL.to_pickle("Outputs/ForS4/scaled_train_with_modifiedNIA_0725.pkl")
scaled_test_DL.to_pickle("Outputs/ForS4/scaled_test_with_modifiedNIA_0725.pkl")

In [ ]:
scaled_train_DL.to_csv("Outputs/ForS4/scaled_train_with_modifiedNIA_0725.csv", index=False)
scaled_test_DL.to_csv("Outputs/ForS4/scaled_test_with_modifiedNIA_0725.csv", index=False)

In [ ]:
# scaled_train.to_pickle("NIAvalueUpdatedVersion_scaled_train.pkl")
# scaled_test.to_pickle("NIAvalueUpdatedVersion_scaled_test.pkl")
scaled_whole_decedents_dataset.to_pickle("Outputs/NIAvalueUpdatedVersion_scaled_whole_decedents_dataset.pkl")
scaled_whole_decedents_dataset.to_csv("Outputs/NIAvalueUpdatedVersion_scaled_whole_decedents_dataset.csv", index=False)

In [ ]:
print(f"The unique number of projid in scaled_whole_decedents_dataset is {scaled_whole_decedents_dataset['projid'].nunique()}")